#### 1.1.1 Online Retail Data Set
The Online Retail Data Set [1] is a dataset made available on the UCI Machine Learning repository. It which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based online retail.

Each of the 541,909 rows contains an item that has been purchased by someone. Items can be grouped into invoices (you can think of these as receipts), where each invoice has been issued for a specific buyer, and can contain multiple items.
The columns contained in the CSV file are the following:
- InvoiceNo: Invoice number. Nominal, a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter “C”, it indicates a cancellation.
- StockCode: Product (item) code. Nominal, a 5-digit integral number uniquely assigned to each distinct product.
- Description: Product (item) name. Nominal.
- Quantity: The quantities of each product (item) per transaction. Numeric.
- InvoiceDate: Invoice Date and time. Numeric, the day and time when each transaction was generated.
- UnitPrice: Unit price. Numeric, Product price per unit in sterling.
- CustomerID: Customer number. Nominal, a 5-digit integral number uniquely assigned to each customer.
- Country: Country name. Nominal, the name of the country where each customer resides.

In [ ]:
# ignore, this DS is different from the CSV
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
# data (as pandas dataframes) 
x = online_retail.data.features 
y = online_retail.data.targets 

display(x)
display(y)

# metadata 
print(online_retail.metadata)

  
# variable information 
print(online_retail.variables) 

In [ ]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.preprocessing import TransactionEncoder

In [ ]:
df = pd.read_csv('../../Dataset/LAB7/online_retail.csv', sep=',')
display(df)

This exercise will work on the Online Retail Data Set. In particular, you will perform data preprocessing on the dataset to extract all itemsets available (**where each itemset is a collection of items contained in a single invoice**). Then, using FP-Growth and Apriori implementations, you will extract a list of frequent itemsets. From those, you will finally extract several different association rules.

#### 1. First, you need to load the dataset into memory.  
Make sure you identify all valid rows. Also consider that rows having an InvoiceNo that starts with C should be discarded, as they indicate that the invoice is about a cancelled purchase.

In [ ]:
df.describe()

- First thing I notice is quantity max and min being -80995.000000 and 80995.000000	 --> how TF can you buy a negative quantity and who the fuck buys 80995 items?
- Same for unitprice -11062.060000	and 38970.000000

In [ ]:
df.info()

- These columns have some missing values:
    - Description  540455 non-null
    - CustomerID   406829 non-null

The columns have different types:
- ID codes
- text
- numbers
- date
- country

In [ ]:
print("Nan value per column:")
for col in df:
    print(f"{col} : {df[col].isna().sum()} / {len(df[col])} --> { ( df[col].isna().sum() / len(df[col]) )*100 }%")

- CustomerID : 135080 / 541909 --> 24.926694334288598%  
A bit frightnening, we need to figure out why.

In [ ]:
for row_idx in df.index:
    nan_per_row = df.iloc[row_idx].isna().sum()
    val_per_row = len(df.iloc[row_idx])
    perc_nan_per_row = nan_per_row / val_per_row
    if perc_nan_per_row > 0.25:
        print(f"The row {row_idx} has {perc_nan_per_row}% Nan values")

# rows are ok

Ok, now we need to answer:
1. CustomerID : 135080 / 541909 --> 24.926694334288598% Nan values  
    - A bit frightnening, we need to figure out why --> maybe I have Nan value for all transitions with a 'C' (cancelled), or maybe not all of them but most of them -> let's check.

---

2. quantity max and min being -80995.000000 and 80995.000000  
    - how TF can you buy a negative quantity and who the fuck buys 80995 items?
3. Same for unitprice -11062.060000	and 38970.000000  

- weird numeric outliers --> PLOT DISTRIBUTION --> **NO**, DISTRIBUTION IS LIKE HOW MANY TRANSACTIONS HAS QUANTITY 6, I WANT TO PLOT THE SINGLE QUANTITY VALUES FROM THE DS --> **SCATTER PLOT**

In [ ]:
# for i,value in enumerate(df['CustomerID'].isna()):
#     if value:
#         print(df.iloc[i])
#         break

# it's quicker if you use the output of .isna(), which are booleans, as index for masking

boolean_mask = df['CustomerID'].isna()
row_Nan_customer_ids = df[boolean_mask]
display(row_Nan_customer_ids)
# 135080 rows has missing CustomerID
# out of these how many have InvoiceNo starting with 'C'?

# for row in row_missing_customer_ids['InvoiceNo']:
#     if row.startswith('C'):
#         print(row)

mask = row_Nan_customer_ids['InvoiceNo'].str.startswith('C')
cancelled_row_Nan_customer_ids = row_Nan_customer_ids['InvoiceNo'][mask]
print(cancelled_row_Nan_customer_ids)
# out of these how many have InvoiceNo starting with 'C'?
# 383


1. CustomerID : 135080 / 541909 --> 24.926694334288598% Nan values  
So, unfortunately it turns out that the Nan values in CustomerID are not associated to cancelled transactions, like at all.
What do I do? keep them? impute them? drop them?

> I'll ignore them out of laziness :)

In [ ]:
# hold on let's see the description

mask = df['CustomerID'].isna()
display(df[mask]['Description'])

# nope nothing in the description about these ghosts

In [ ]:
display(df.head(5))

In [ ]:
# plot a histogram --> NO, HISTOGRAMS ARE TO PLOT THE DISTRIBUTION = HOW MANY TIMES A VALUE OCCURS IN THE DS, like how many times the value 6 occurs
# --> I WANT TO PLOT THE SINGLE ACTUAL VALUES --> **SCATTER PLOT**
    # plot transactions on the x
    # plot quantity on the y


plt.figure(figsize=(8,6))

sns.scatterplot(x = df.index, y = df['Quantity'])
plt.title('Spot quantity outliers / anomalies')
plt.xlabel('Transactions')
plt.ylabel('Quantity values')

plt.show()

#### Goddamn this is weird man
Maybe the quantities are negative when the transaction is cancelled?  
Well in any case I have to remove the cancelled transaction, drop them and plot/explore again the quantities and maybe the negative quantities will disappear.  

In [ ]:
display(df)

In [ ]:
bool_mask_cancelled_ransactions = df['InvoiceNo'].str.startswith('C')
cancelled_ransactions = df[bool_mask_cancelled_ransactions]
display(cancelled_ransactions)
# yep, cancelled transactions have negative quantities :)

cleaned_df = df.drop(index=cancelled_ransactions.index)
display(cleaned_df)

mask = cleaned_df['InvoiceNo'].str.startswith('C')
display(cleaned_df[mask])
# this is empty, nice, just a sanity check
# now that I dropped all cancelled trasactions, will all the negative quantities disappear?

mask = cleaned_df['Quantity'] < 0
negative_quantities = cleaned_df[mask]
display(negative_quantities)
display(negative_quantities['Description'].value_counts())
# it still has negative quantities, this DS is a mystery man
# AAAAAHHHH, read the descriptions HAHAHAHAHAHAH --> lost, missing, smashed :)

#### Now quantity values should be fixed

In [ ]:
cleaned_df.describe()

#### Fuck this, I'll manually check these anomalies:

In [ ]:
mask = cleaned_df['Quantity'] == -9600.000000
display(cleaned_df[mask])

# so it's actually NOT an anomaly

In [ ]:
mask = cleaned_df['Quantity'] == 80995.000000
display(cleaned_df[mask])

In [ ]:
mask = cleaned_df['UnitPrice'] == -11062.060000
display(cleaned_df[mask])

# AAAAHHHHHHH, okok

In [ ]:
mask = cleaned_df['UnitPrice'] == 13541.330000
display(cleaned_df[mask])

In [ ]:
plt.figure(figsize=(10,10))

plt.subplot(2,2,1)
sns.scatterplot(x = cleaned_df.index, y = cleaned_df['Quantity'])
plt.title('New quantities')
plt.xlabel('Transactions')
plt.ylabel('Quantity values')

plt.subplot(2,2,2)
sns.scatterplot(x = cleaned_df.index, y = cleaned_df['UnitPrice'])
plt.title('New unit price')
plt.xlabel('Transactions')
plt.ylabel('Unit price values')

plt.tight_layout()
plt.show()

#### SO:
I did a ton of data exploration for absolutely nothing :)  
- It's all ok, the missing customer ids are because of weird stuff, like paying debts, fees, breaking stuff, ecc.
- negative quantities, some of them got dropped by dropping the cancelled transactions + those negative quantities that still survived are normal: missing stuff, breaking stuff, paying debts
- the huge negative and positive anomalies in quantities are normal too
- same here

So we can finally start the lab :)

---

In [ ]:
df = cleaned_df
display(df)

2. Now that you have a dataset of items, you should aggregate it at an “invoice” level. For each invoice (identified by InvoiceNo) there can be multiple items (from multiple rows) in the dataset. For each invoice, create an array of all items associated with it. For the example invoice presented in 1.1.1, you want to build the following array:

```text
array ([' GARDENERS KNEELING PAD KEEP CALM ',' HOT WATER BOTTLE KEEP CALM ',' DOORMAT KEEP CALM AND COME IN ' ])
```

In [ ]:
# a dictionary is a better data structure but later on I need to use a fucking retarded encoder --> use 2D arrays / 2D list
option_1 = False
if option_1:
    transactions = {}
    grouped = df.groupby('InvoiceNo')

    for id,value in grouped:
        transactions[id] = value['Description'].tolist()

    # transactions is humongous, I cannot visualize it because the print fails
    # use this escamotage
    keys = list(transactions.keys())
    for k in keys[:15]:
        print(f"{k} : {transactions[k]}")

option_2 = True
# manipulate data using lists, no need for arrays
if option_2:
    transactions = []
    grouped = df.groupby('InvoiceNo')
    for id, value in grouped:
        transactions.append(list(value['Description'].dropna()))

    # sanity check
    sanity_check = True
    if sanity_check:
        print(transactions[:][:5])      # printing items bought in 5 transactions, each row = transactions
        for id, value in grouped:
            print()
            print(id)
            display(value)
            break

# array_transactions = np.array(transactions)
# You cannot use arrays, because each transaction has a different number of items in it,
# for each transaction I don't buy the alwasy same number of items
# --> keep lists

You should now have an array (one for each invoice) of arrays (each array containing the items bought for that invoice).

- Now, we need to convert this into a matrix form. Of the many possible formats, we will use the one expected by the Mixtend library, which is as follows:
    - Given an ordered array of M possible items (in this case, all possible products that can be bought),
    - and given N itemsets (in this case, invoices),
    - we should build a matrix of N rows and M columns --> N invoices x M possible products/items
        - SO, the element at the ith row = bought in the ith invoice and jth column = the element bought is the jth item, should be 1 if the ith itemset (invoice) contains the jth item (product), O otherwise.

        - For the following example:

a, b, c  
b, c  
a, c, d  
a, b  

The list of all possible items is [a, b, c, d]. As such, the matrix that we will build is the following:
1 1 1 0  
0 1 1 0  
1 0 1 1  
1 1 0 0  

Once we have defined this matrix, we can convert it to a DataFrame.

---
```python
This is writte in the official documentation of Mlxtend:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)
```

> # BUT the TransactionEncoder expects as imput a 2D array, not my gorgeuous dictionary :(
So I have to go from this:  
{  
    536365 : ['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN', 'CREAM CUPID HEARTS COAT HANGER', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'RED WOOLLY HOTTIE WHITE HEART.', 'SET 7 BABUSHKA NESTING BOXES', 'GLASS STAR FROSTED T-LIGHT HOLDER']  
    ....  
}  

To this  
[  
    ['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN', 'CREAM CUPID HEARTS COAT HANGER', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'RED WOOLLY HOTTIE WHITE HEART.', 'SET 7 BABUSHKA NESTING BOXES', 'GLASS STAR FROSTED T-LIGHT HOLDER'] --> which is transaction 1,  
    [transaction 2],  
    ...  
]  

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, apriori, association_rules

In [ ]:
print(df['Description'].value_counts())

print()

print(df['Description'].unique())
print(len(df['Description'].unique()))

In [ ]:
# later, in the following cell I'm having trouble doing the fit_transform because of data type inconsistency str and float ???
# let's inspect that
for transaction in transactions:
    for item in transaction:
        if type(item) != str:
            print(item)
# fuuuuuuuuuuck I have nan values
# you dumbass you already knew this since the beginning when you did the data inspection, you forgot >:(
# drop or impute those fuckers?
# they're a small % --> drop them bitches

- Later when I'll recall the TransactionEncoder, it will not be happy because it says TypeError: '<' not supported between instances of 'float' and 'str' --> I checked which values were not strings and I found out that description has some nan values  
- I correctly insulted myself saying that I'm a dumbass because you already knew this:

Nan value per column:  
InvoiceNo : 0 / 541909 --> 0.0%  
StockCode : 0 / 541909 --> 0.0%  
Description : 1454 / 541909 --> 0.2683107311375157%  
Quantity : 0 / 541909 --> 0.0%  
InvoiceDate : 0 / 541909 --> 0.0%  
UnitPrice : 0 / 541909 --> 0.0%  
CustomerID : 135080 / 541909 --> 24.926694334288598%  
Country : 0 / 541909 --> 0.0%  

- I didn't do anything because the % was small and thouth I could ignore it, but now I have to deal with it because TransactionEncoder is being a bitch
- since the % is very small drop those nan values, I changed this:  

list(value['Description']**.dropna()**)

In [ ]:
OHencoder = TransactionEncoder()
encoded_array = OHencoder.fit_transform(transactions)
print(encoded_array.shape)

ohe_df = pd.DataFrame(encoded_array, columns=OHencoder.columns_)
display(ohe_df)

#### 4. With the df that you defined in the previous exercise, you can now use the fp_growth function.  
This function, which is described in the detail in the official documentation. The first argument required is the previously built DataFrame, df. The second is the minimum support (minsup), i.e., the minimum fraction of the entire dataset in which the itemset must appear for it to be considered “frequent”. Tryusingdifferentvaluesofminsup,suchas. 5,0.1,0.05, 0.02,0.01. Howmanyresults do you obtain as minsup varies? 

In [ ]:
min_supports=[0.5, 0.1, 0.05, 0.02, 0.01]
for min_support in min_supports:
    frequent_itemset = fpgrowth(ohe_df, min_support=min_support, use_colnames=True)
    print(f"For minsupport = {min_support} there are {len(frequent_itemset)} frequent itemsets:")
    display(frequent_itemset)
    print()
    

#### 5. Consider the itemsets extracted for minsup = 0.02. How many items are contained? Which ones would you consider to be the most useful?

In [ ]:
frequent_itemsets = fpgrowth(ohe_df, min_support=0.02, use_colnames=True)
display(frequent_itemsets)

# sorted_support_series = frequent_itemset['support'].sort_values(ascending = False)
# sorted_support_idx = sorted_support_series.index
# print(sorted_support_idx)
# display(frequent_itemset.iloc[sorted_support_idx])

sorted_frequent_itemsets = frequent_itemsets.sort_values(by ='support', ascending = False)
display(sorted_frequent_itemsets)
print(len(sorted_frequent_itemsets))


#### 6. Select one of the frequent itemsets returned by fpgrowth. Use this itemset to extract the relevant association rules. For these rules, compute their confidence.

**association_rules()**  
1.	Non ha senso passare un singolo itemset al metodo association_rules(). Invece, devi passare tutti gli itemsets frequenti che hai ottenuto con fpgrowth, in modo da generare tutte le regole basate su quelli.  
2.	L’istruzione “Seleziona uno dei frequent itemsets restituiti da fpgrowth” probabilmente ti chiede di analizzare le regole derivanti da uno specifico itemset (dopo aver generato tutte le regole), per calcolare la confidenza di quelle regole, o forse di esplorare solo le regole che contengono un determinato itemset. Ma non ha senso limitarsi a passare un solo itemset al association_rules().  

In [ ]:
rules = association_rules(df=frequent_itemsets, metric='confidence', min_threshold=0.2)
display(rules.head(5))

print('-------------------')

# sort on support and then confidence.
sorted_support_rules = rules.sort_values(by='support', ascending=False)
display(sorted_support_rules.head(5))

print('-------------------')

sorted_confidence_rules = rules.sort_values(by='confidence', ascending=False)
display(sorted_confidence_rules.head(5))

#### 7. Extract the association rules from the frequent itemsets extracted with minsup = 0.01.
You can find the documentation for association_rules() on the official documentation. You can use the confidence as the metric to identify the rules, and a minimum threshold of 0.85 (feel free to vary these values and observe how the results vary).

In [ ]:
frequent_itemsets = fpgrowth(ohe_df, min_support=0.01, use_colnames=True)

rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.85)
display(rules)